# Reading data with coffea NanoEvents

NanoEvents is a Coffea utility to wrap flat nTuple structures (such as the CMS [NanoAOD](https://www.epj-conferences.org/articles/epjconf/pdf/2019/19/epjconf_chep2018_06021.pdf) format) into a single awkward array with appropriate object methods (such as Lorentz vector methods$^*$), cross references, and nested objects, all accessed in virtual$\dagger$ mode from the source ROOT TTree via uproot. The interpretation of the TTree data is configurable via [schema objects](https://coffea-hep.readthedocs.io/en/latest/modules/coffea.nanoevents.html#classes), which are community-supplied  for various source file types. These schema objects allow a richer interpretation of the file contents than the `uproot` methods. Currently available schemas include:

   - `BaseSchema`, which provides a simple representation of the input TTree, where each branch is available verbatim as `events.branch_name`.  Any branches that uproot supports at "full speed" (i.e. that are fully split and either flat or single-jagged) can be read by this schema;
   - `NanoAODSchema`, which is optimized to provide all methods and cross-references in CMS NanoAOD format;
   - `PFNanoAODSchema`, which builds a double-jagged particle flow candidate colllection `events.jet.constituents` from compatible PFNanoAOD input files;
   - `TreeMakerSchema` which is designed to read TTrees made by [TreeMaker](https://github.com/TreeMaker/TreeMaker), an alternative CMS nTuplization format;
   - `PHYSLITESchema`, for the ATLAS DAOD_PHYSLITE derivation, a compact centrally-produced data format similar to CMS NanoAOD; and
   - `DelphesSchema`, for reading Delphes fast simulation [nTuples](https://cp3.irmp.ucl.ac.be/projects/delphes/wiki/WorkBook/RootTreeDescription).

**We welcome contributions for new schemas, and can assist with the design of them.**

As of v2025.11.0, Coffea supports reading RNTuples. Since RNTuples are columnar by design, the contained data can already have the structure that Coffea schemas aim to implement. In fact, since they can also contain metadata in the form of RNTuple attributes, it might be possible to embed the necessary metadata to make some schemas unnecessary. However, RNTuple implementations from different experiments are not yet finalized, so we will have to see what happens when the dust settles. For this training, we will stick to using TTrees since this is still the format most people will be using in the short term.

$^*$ Vector methods are provided by the dedicated scikit-hep [vector](https://vector.readthedocs.io/en/latest/) library, which provides a more rich feature set than the deprecated `coffea-vector`.

$\dagger$ virtual access refers to only fetching the needed data from the (possibly remote) file when you're ready to compute the entire set of outputs. Until then, a minimal amount of metadata is used to trace the operations to eventually be performed (frequently called "lazily" "booking" operations in RDataFrame's approach to this)


In this demo, we will use NanoEvents to read a small CMS NanoAOD sample. The events object can be instantiated as follows:

# NanoAOD Example

In [ ]:
import awkward as ak
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

NanoAODSchema.warn_missing_crossrefs=False

fname = "../data/nano_dy.root"
access_log = []
events = NanoEventsFactory.from_root(
    {fname: "Events"}, # We pass a dictionary of {filename1: treename1, filename2: treename2, ...} to load from
    schemaclass=NanoAODSchema,
    metadata={"dataset": "DYJets"},
    mode="eager",
    access_log=access_log,
).events()

In the factory constructor, we also pass the desired schema version (the latest version of NanoAOD can be built with `schemaclass=NanoAODSchema`) for this file and some extra metadata that we can later access with `events.metadata`. In a later example, we will show how to set up this metadata in coffea processors where the `events` object is pre-created for you. Consider looking at the [from_root](https://coffea-hep.readthedocs.io/en/latest/api/coffea.nanoevents.NanoEventsFactory.html#coffea.nanoevents.NanoEventsFactory.from_root) class method to see all optional arguments.

The `events` object acts like an Awkward array, which at its top level is a record array with one record for each "collection", where a collection is a grouping of fields (TBranches) based on the naming conventions of [NanoAODSchema](https://coffea-hep.readthedocs.io/en/latest/api/coffea.nanoevents.NanoAODSchema.html). For example, in the file we opened, the branches:
```
Generator_binvar
Generator_scalePDF
Generator_weight
Generator_x1
Generator_x2
Generator_xpdf1
Generator_xpdf2
Generator_id1
Generator_id2
```
are grouped into one sub-record named `Generator` which can be accessed using either getitem or getattr syntax, i.e. `events["Generator"]` or `events.Generator`. e.g.

In [ ]:
events.Generator.id1

In [ ]:
# all names can be listed with:
events.Generator.fields

In CMS NanoAOD, each TBranch has a self-documenting help string embedded in the title field, which is carried into the NanoEvents, e.g. executing the following cell should produce a help pop-up:
```
Type:            Array
String form:     [1, -1, -1, 21, 21, 4, 2, -2, 2, 1, 3, 1, ... -1, -1, 1, -2, 2, 1, 2, -2, -1, 2, 1]
Length:          40
File:            ~/src/awkward-1.0/awkward1/highlevel.py
Docstring:       id of first parton
Class docstring: ...
```
where the `Docstring` shows information about the content of this array.

In [ ]:
events.Generator.id1?

Based on a collection's name or contents, some collections acquire additional _methods_, which are extra features exposed by the code in the mixin classes of the `coffea.nanoevents.methods` modules. For example, although `events.GenJet` has the fields:

In [ ]:
events.GenJet.fields

we can access additional attributes associated to each generated jet by virtue of the fact that they can be interpreted as [Lorentz vectors](https://coffea-hep.readthedocs.io/en/latest/api/coffea.nanoevents.methods.vector.LorentzVector.html#coffea.nanoevents.methods.vector.LorentzVector):

In [ ]:
events.GenJet.energy

We can call more complex methods, like computing the distance $\Delta R = \sqrt{\Delta \eta^2 + \Delta \phi ^2}$ between two LorentzVector objects:

In [ ]:
# find distance between leading jet and all electrons in each event
dr = events.Jet[:, 0].delta_r(events.Electron)
dr

In [ ]:
# find minimum distance
drmin = ak.min(dr, axis=1)
drmin

In [ ]:
# a convenience method for this operation on all jets is available
events.Jet.nearest(events.Electron)

The assignment of methods classes to collections is done inside the schema object during the initial creation of the array, governed by the awkward array's `__record__` parameter and the associated behavior. See [ak.behavior](https://awkward-array.readthedocs.io/en/latest/ak.behavior.html) for a more detailed explanation of array behaviors.

Additional methods provide convenience functions for interpreting some branches, e.g. CMS NanoAOD packs several jet identification flag bits into a single integer, `jetId`. By implementing the bit-twiddling in the [Jet mixin](https://github.com/CoffeaTeam/coffea/blob/7045c06b9448d2be4315e65d432e6d8bd117d6d7/coffea/nanoevents/methods/nanoaod.py#L279-L282), the analsyis code becomes more clear:

In [ ]:
print(events.Jet.jetId)
print(events.Jet.isTight)

We can also define convenience functions to unpack and apply some mask to a set of flags, e.g. for generated particles:

In [ ]:
print(f"Raw status flags: {events.GenPart.statusFlags}")
events.GenPart.hasFlags(['isPrompt', 'isLastCopy'])

CMS NanoAOD also contains pre-computed cross-references for some types of collections. For example, there is a TBranch `Electron_genPartIdx` which indexes the `GenPart` collection per event to give the matched generated particle, and `-1` if no match is found. NanoEvents transforms these indices into an awkward _indexed array_ pointing to the collection, so that one can directly access the matched particle using getattr syntax:

In [ ]:
events.Electron.matched_gen.pdgId

In [ ]:
events.Muon[ak.num(events.Muon)>0].matched_jet.pt

For generated particles, the parent index is similarly mapped:

In [ ]:
events.GenPart.parent.pdgId

In addition, using the parent index, a helper method computes the inverse mapping, namely, `children`. As such, one can find particle siblings with:

In [ ]:
events.GenPart.parent.children.pdgId
# notice this is a doubly-jagged array

Since often one wants to shortcut repeated particles in a decay sequence, a helper method `distinctParent` is also available. Here we use it to find the parent particle ID for all prompt electrons:

In [ ]:
events.GenPart[
    (abs(events.GenPart.pdgId) == 11)
    & events.GenPart.hasFlags(['isPrompt', 'isLastCopy'])
].distinctParent.pdgId

Events can be filtered like any other awkward array using boolean fancy-indexing

In [ ]:
mmevents = events[ak.num(events.Muon) == 2]
zmm = (mmevents.Muon[:, 0] + mmevents.Muon[:, 1])
zmm.mass

In [ ]:
# a convenience method is available to sum vectors along an axis:
mmevents.Muon.sum(axis=1).mass

As expected for this sample, most of the dimuon events have a pair invariant mass close to that of a Z boson. But what about the last event? Let's take a look at the generator information:

In [ ]:
print(mmevents[-1].Muon.matched_gen.pdgId)
print(mmevents[-1].Muon.matched_gen.hasFlags(["isPrompt"]))

So they are real generated muons, but they are not prompt (i.e. from the initial decay of a heavy resonance)

Let's look at their parent particles:

In [ ]:
mmevents[-1].Muon.matched_gen.parent.pdgId

aha! They are muons coming from tau lepton decays, and hence a fair amount of the Z mass is carried away by the neutrinos:

In [ ]:
print(mmevents.Muon.matched_gen.sum().mass[-1])
print(mmevents.Muon.matched_gen.parent.sum().mass[-1])

One can assign new variables to the arrays, with some caveats:

 * Assignment must use setitem (`events["path", "to", "name"] = value`)
 * Assignment to a sliced `events` won't be accessible from the original variable
 * New variables are not visible from cross-references

In [ ]:
mmevents["Electron", "myvariable"] = mmevents.Electron.pt + zmm.mass
mmevents.Electron.myvariable

NanoEvents supports reading lazily in "virtual" mode. In that mode, branches will only be loaded from the root file when an explicit computation needs them.

In [ ]:
fname = "../data/nano_dy.root"
access_log = []
events = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset": "DYJets"},
    mode="virtual",
    access_log=access_log,
).events()

In [ ]:
print(events.Jet.nearest(events.Electron))
print(access_log)

In [ ]:
events.Jet.jetId

In [ ]:
events.Jet.isTight

In [ ]:
access_log

In [ ]:
events.Jet.jetId

In [ ]:
mmevents = events[ak.num(events.Muon) == 2]
zmm = (mmevents.Muon[:, 0] + mmevents.Muon[:, 1])
zmm.mass

In [ ]:
access_log

# PHYSLite Example

In [ ]:
import awkward as ak
import numpy as np
from coffea.nanoevents import NanoEventsFactory, PHYSLITESchema
import numpy as np
import matplotlib.pyplot as plt
import hist
import warnings

Let's load a PHYSLITE file containing simulated event data describing the production the Higgs boson and its decay via a pair of Z bosons to a four lepton final state. Those data are part of the recent ATLAS open data release and can be found at DOI:[10.7483/OPENDATA.ATLAS.Z2J9.709J](http://doi.org/10.7483/OPENDATA.ATLAS.Z2J9.709J).

More information about our data release at [this article](https://atlas.cern/Updates/News/Open-Data-Research) and the ATLAS open data portal: https://opendata.atlas.cern/

You can download those files at your local machine or you can stream them directly. For this demo we will use the [XCache](https://slateci.io/XCache/) service of the UChicago Analysis Facility.

In [ ]:
import urllib, requests
from pathlib import Path

url = "https://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000001.pool.root.1"
file_path = "/tmp/DAOD_PHYSLITE.38191712._000001.pool.root.1"

# Download the file
if not Path(file_path).exists():
    with requests.get(url, stream=True, verify=False) as response:
        response.raise_for_status()
        with open(file_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
else:
    print(f"File already downloaded: {file_path}")

To make the loading lighter, we will use the `filter_name` function, in order to load only the variables needed. Then we will use `coffea`'s `NanoEventsFactory.from_root` to load the file. The variables are grouped in data structures called "containers" and the convention to reference them is `<ContainerName>.<VariableName>`. Consider looking at the [`from_root`](https://coffeateam.github.io/coffea/api/coffea.nanoevents.NanoEventsFactory.html#coffea.nanoevents.NanoEventsFactory.from_root) class method to see all optional arguments.

All the containers and variables stored in the open data PHYSLITE files are listed in: https://atlas-physlite-content-opendata.web.cern.ch/. For your reference, the equivalent internal webpage for general PHYSLITE files is: https://atlas-physlite-content.web.cern.ch/. 

_Some warnings will appear as we're still working with `uproot` and the `PHYSLITESchema` to correctly interpret all the variables stored in PHYSLITE._

In [ ]:
warnings.filterwarnings(
    "ignore",
    message="Skipping ",
    category=UserWarning,
)

events = NanoEventsFactory.from_root(
    {file_path: "CollectionTree"}, # all the event variables are stored in the TTree called CollectionTree
    schemaclass=PHYSLITESchema, # tell NanoEventsFactory.from_root that you read a PHYSLITE file
    mode="virtual", # use "eager" if you want to load all the events on the spot
).events()

The `events` object is an awkward array, which at its top level is a record array* with one record for each "collection", where a collection is a grouping of objects of the seme type, based on the naming conventions of `PHYSLITESchema`. We try to correspond the collections in awkward arrays with the containers found in PHYSLITE.

\* _Record array_ a is a specialized type of array that allows you to store elements with mixed data types that can be accessed using attribute notation, e.g. `array.x` [[1](https://numpy.org/doc/stable/reference/generated/numpy.recarray.html)], [[2](https://awkward-array.org/doc/main/reference/generated/ak.Record.html)].

For example, in the file we opened there are the collections:

In [ ]:
events.fields

For each collection, we can list the variables that have been loaded (TBranches). For example, let's look at the variables of the `Electrons`:

In [ ]:
events.Electrons.fields

The data in a PHYSLITE file are highly structured but jagged. In other words, in each event there are different number of objects in each collection, thus the awkward arrays are not rectilinear but _awkward_. We can see that if we actually request to compute the objects of the `Electrons` collection.

In [ ]:
ak.materialize(events.Electrons)

## Slicing

One of the most common operations you will found yourself doing is event selection or array _slicing_. We can practice this by first defining an awkward (boolean) array by a logical operation. Then we can use this array as an argument to slice our events.

For example, let's request events with at least two electrons:

In [ ]:
# this cell might fail on the first time -- if so, run twice or restart the kernel

# define boolean array
selection_2e = ak.num(events.Electrons, axis=1) > 1

# print the selected events
events[selection_2e]

Notice how the number of events in the awkward array has been reduced.

In [ ]:
# print the second of the selected events
events[selection_2e][1]

In [ ]:
# print the leading electron pt of the selected events
events.Electrons[selection_2e][:, 0].pt # in MeV
# events[selection_2e][:, "Electrons"][:, "pt"][:, 0] # another uglier way to write this

Next, we will demonstrate something more complicated. This is an example of the power of the schemas along with vector operations*. We will use a utility method [`delta_r`](https://coffeateam.github.io/coffea/api/coffea.nanoevents.methods.vector.LorentzVector.html#coffea.nanoevents.methods.vector.LorentzVector.delta_r) to calculate the distance $\Delta R = \sqrt{\Delta\eta^2+\Delta\phi^2}$ between two LorentzVector objects. In particular, we will calculate the distance between the leading and sub-leading electrons in each event. Of course, we need to make use we use events that have at least two electrons, thus we will utilize the `selection_2e` from above.

To plot a histogram we will just use `matplotlib` for the moment. You might also want to try the [`hist`](https://github.com/scikit-hep/hist) library, which you will learn about at the [Introduction to hist](https://indico.cern.ch/event/1376945/timetable/?view=standard#33-introduction-to-hist) lecture.

_\* `coffea` has switched from the internal [`coffea.nanoevents.methods.vector`](https://coffeateam.github.io/coffea/modules/coffea.nanoevents.methods.vector.html) to the [`vector`](https://github.com/scikit-hep/vector) library, if you are on a sufficiently newer coffea release

In [ ]:
# distance between leading and sub-leading electron in every event
dr = events.Electrons[selection_2e][:, 0].delta_r(events.Electrons[selection_2e][:, 1])

# plot
plt.hist(dr, bins=50, range=(0, 5))
plt.xlabel(r"$\Delta R(e_0, e_1)$")
plt.ylabel("Events")
plt.show()

<details>
<summary><b>Hint</b></summary>

Use the [mass](https://coffeateam.github.io/coffea/api/coffea.nanoevents.methods.vector.LorentzVector.html#coffea.nanoevents.methods.vector.LorentzVector.mass) attribute.
</details>

In [ ]:
# your code here

<details>
<summary><b>Answer</b></summary>

```python
# define boolean array to select events with 4 electrons
selection_4e = ak.num(events.Electrons, axis=1) == 4

# calculate the invariant mass of the 4 electrons
mass = (events.Electrons[selection_4e][:, 0] + 
        events.Electrons[selection_4e][:, 1] + 
        events.Electrons[selection_4e][:, 2] + 
        events.Electrons[selection_4e][:, 3]).mass

# plot
plt.hist(mass/1000, bins=10, range=(100, 150))
plt.xlabel(r"$m(4e)$ [GeV]")
plt.ylabel("Events")
plt.show()
```
</details>

## Element Links

Some collections contain information which is contextually relevant to another collection. For example, in PHYSLITE files we store the `BTagging_AntiKt4EMPFlow` collection which contains information regarding the probabilities of the flavour origin of the objects of the `Jets` collection.

The information is referenced and accessed by the `ElementLinks`. `ElementLinks` are like pointers that help point to specific objects in a collection. They have a structure similar to a Python dictionary with a key (`m_persKey`) to identify the collection they point to and a value (`m_persIndex`) to identify the index of the element in that collection. 

On the above example, each object of the `Jets` collection (i.e. each jet) holds a variable called `btaggingLink`, which points to the objects of the `BTagging_AntiKt4EMPFlow` collection. The index is used to identify which exactly `BTagging_AntiKt4EMPFlow` object corresponds to each jet.

In [ ]:
events.Jets.btaggingLink.fields

### Naive Linking

We're currently working with `coffea` and the `PHYSLITESchema` to enable automatic cross-reference (_ElementLinking_) between collections. However, in simple cases we can achieve the linking, without actually using the `ElementLinks`.

When there is **one-to-one** correspondence between the objects of two collections, we can link those without the need to resolve the actual links. For our example, each object of the `Jets` collection is linked to one object of the `BTagging_AntiKt4EMPFlow` collection.

We will write a function to perform that linking. We will use the [DL1dv01](https://ftag.docs.cern.ch/algorithms/taggers/dl1/) flavour-tagging algorithm and information about its calibration for b-tagging can be found [here](https://ftag.docs.cern.ch/recommendations/algs/r22-preliminary/#working-point-definition-for-dl1dv01).

In [ ]:
def calculate_jets_DL1dv01(events):
    
    BTagging = events.BTagging_AntiKt4EMPFlow
    
    f_c = 0.018
    DL1dv01 = BTagging.DL1dv01_pb/(f_c*BTagging.DL1dv01_pc + (1-f_c)*BTagging.DL1dv01_pu)
    DL1dv01 = np.log(DL1dv01)

    return DL1dv01

We will use the above function to assign a new variable to the `Jets` collection. One can assign new variables to the arrays, with some caveats:

* Assignment must use `events["path", "to", "name"] = value`.
* Assignment to a sliced events won't be accessible from the original variable.

In [ ]:
# assign new variable to the collection
events['Jets', 'DL1dv01'] = calculate_jets_DL1dv01(events)

events.Jets.fields

Let's finally use the new variable to make a selection. We will plot the leading jet transverse momentum ($p_T$) for events that have at least one b-tagged jet at the 77% efficiency working point.

In [ ]:
# boolean array -- events with at least one b-jet at 77% WP
selection_1bjet = ak.sum(events.Jets.DL1dv01 > 2.456, axis=-1) > 0

# plot the leading jet pt
plt.hist(events.Jets[selection_1bjet][:, 0].pt/1000, bins=50, range=(0, 500))
plt.xlabel(r"$p_T(j_0)$ [GeV]")
plt.ylabel("Events")
plt.show()

In [ ]:
# distance between leading and sub-leading electron in the same selected events
dr1b = events.Electrons[selection_2e & selection_1bjet][:, 0].delta_r(events.Electrons[selection_2e & selection_1bjet][:, 1])

# plot
plt.hist(dr1b, bins=50, range=(0, 5))
plt.xlabel(r"$\Delta R(e_0, e_1)$")
plt.ylabel("Events")
plt.show()

# Processors

Coffea relies mainly on [uproot](https://github.com/scikit-hep/uproot) to provide access to ROOT files for analysis.
As a usual analysis will involve processing tens to thousands of files, totalling gigabytes to terabytes of data, there is a certain amount of work to be done to build a parallelized framework to process the data in a reasonable amount of time. 

Since the beginning a `coffea.processor` module was provided to encapsulate the core functionality of the analysis, which could be run locally or distributed via a number of Executors. This allowed users to worry just about the actual analysis code and not about how to implement efficient parallelization, assuming that the parallization is a trivial map-reduce operation (e.g. filling histograms and adding them together).

Let's start by writing a simple processor class that reads some CMS open data and plots a dimuon mass spectrum.
We'll start by copying the [ProcessorABC](https://coffea-hep.readthedocs.io/en/latest/api/coffea.processor.ProcessorABC.html#coffea.processor.ProcessorABC) skeleton and filling in some details:

 * Adding a new histogram for $m_{\mu \mu}$
 * Building a [Candidate](https://coffea-hep.readthedocs.io/en/latest/api/coffea.nanoevents.methods.candidate.PtEtaPhiMCandidate.html#coffea.nanoevents.methods.candidate.PtEtaPhiMCandidate) record for muons, since we will read it with `BaseSchema` interpretation (the files used here could be read with `NanoAODSchema` but we want to show how to build vector objects from other TTree formats) 
 * Calculating the dimuon invariant mass

In [ ]:
from coffea import processor
from coffea.nanoevents.methods import candidate

class MyProcessor(processor.ProcessorABC):
    def __init__(self):
        pass

    def process(self, events):
        dataset = events.metadata['dataset']
        muons = ak.zip(
            {
                "pt": events.Muon_pt,
                "eta": events.Muon_eta,
                "phi": events.Muon_phi,
                "mass": events.Muon_mass,
                "charge": events.Muon_charge,
            },
            with_name="PtEtaPhiMCandidate",
            behavior=candidate.behavior,
        )

        h_mass = (
            hist.Hist.new
            .StrCat(["opposite", "same"], name="sign")
            .Log(1000, 0.2, 200., name="mass", label=r"$m_{\mu\mu}$ [GeV]")
            .Int64()
        )

        cut = (ak.num(muons) == 2) & (ak.sum(muons.charge, axis=1) == 0)
        # add first and second muon in every event together
        dimuon = muons[cut][:, 0] + muons[cut][:, 1]
        h_mass.fill(sign="opposite", mass=dimuon.mass)

        cut = (ak.num(muons) == 2) & (ak.sum(muons.charge, axis=1) != 0)
        dimuon = muons[cut][:, 0] + muons[cut][:, 1]
        h_mass.fill(sign="same", mass=dimuon.mass)

        return {
            dataset: {
                "entries": len(events),
                "mass": h_mass,
            }
        }
    
    def postprocess(self, accumulator):
        pass

If we were to just use bare uproot to execute this processor, we could do that with the following example, which:

 * Opens a CMS open data file
 * Creates a NanoEvents object using `BaseSchema` (roughly equivalent to the output of reading with plain `uproot`)
 * Creates a `MyProcessor` instance
 * Runs the `process()` function, which returns our accumulators

In [ ]:
from coffea.nanoevents import BaseSchema
# eospublic xrootd utl
filename = "root://eospublic.cern.ch//eos/root-eos/cms_opendata_2012_nanoaod/Run2012B_DoubleMuParked.root"
# public https url
# filename = "https://xrootd-local.unl.edu:1094//store/user/AGC/zzto4mu/Run2012B_DoubleMuParked.root"
access_log = []
events = NanoEventsFactory.from_root(
    {filename: "Events"},
    entry_stop=500_000,
    metadata={"dataset": "DoubleMuon"},
    schemaclass=BaseSchema,
    mode="virtual",
    access_log=access_log,
).events()
p = MyProcessor()
out = p.process(events)
out, access_log

In [ ]:
%%time
out = p.process(events)
out

In [ ]:
access_log

In [ ]:
fig, ax = plt.subplots()
out["DoubleMuon"]["mass"].plot1d(ax=ax)
ax.set_xscale("log")
ax.legend(title="Dimuon charge")
plt.show()

In [ ]:
initial_fileset = {
    "DoubleMuon": {
        "files": {
            # eospublic xrootd utl
            "root://eospublic.cern.ch//eos/root-eos/cms_opendata_2012_nanoaod/Run2012B_DoubleMuParked.root": "Events",
            # public https url
            # "https://xrootd-local.unl.edu:1094//store/user/AGC/zzto4mu/Run2012B_DoubleMuParked.root": "Events",
        },
        "metadata": {
            "is_mc": False,
        },
    },
    "ZZ to 4mu": {
        "files": {
            # eospublic xrootd utl
            "root://eospublic.cern.ch//eos/root-eos/cms_opendata_2012_nanoaod/ZZTo4mu.root": "Events",
            # public https url
            # "https://xrootd-local.unl.edu:1094//store/user/AGC/zzto4mu/ZZTo4mu.root": "Events",
        },
        "metadata": {
            "is_mc": True,
        }
    }
}

In [ ]:
%%time
iterative_run = processor.Runner(
    executor = processor.IterativeExecutor(compression=None),
    schema=BaseSchema,
    maxchunks=3,
    savemetrics=True,
)

out, metrics = iterative_run(
    initial_fileset,
    processor_instance=MyProcessor(),
)

In [ ]:
fig, ax = plt.subplots()
out["DoubleMuon"]["mass"].plot1d(ax=ax)
ax.set_xscale("log")
ax.legend(title="Dimuon charge")
plt.show()

Now, if we want to use more than a single core on our machine, we simply change `IterativeExecutor` for `FuturesExecutor`, which uses the python `concurrent.futures` standard library. We can then set the most interesting argument to the `FuturesExecutor`: the number of cores to use.

In [ ]:
%%time
futures_run = processor.Runner(
    executor = processor.FuturesExecutor(workers=4, compression=None),
    schema=BaseSchema,
    savemetrics=True,
)

out2, metrics2 = futures_run(
    initial_fileset,
    processor_instance=MyProcessor()
)

In [ ]:
fig, ax = plt.subplots()
out2["DoubleMuon"]["mass"].plot1d(ax=ax)
ax.set_xscale("log")
ax.legend(title="Dimuon charge")
plt.show()

# Scaleout

In [ ]:
import corrections
class MyZPeak(processor.ProcessorABC):
    def __init__(self):
        pass

    def process(self, events):
        dataset = events.metadata['dataset']
        isRealData = "genWeight" not in events.fields
        sumw = 0. if isRealData else ak.sum(events.genWeight, axis=0)
        cutflow = {"start": ak.num(events, axis=0)}
        
        if isRealData:
            events = events[
                corrections.lumimask(events.run, events.luminosityBlock)
            ]
            cutflow["lumimask"] = ak.num(events, axis=0)
    
        events["goodmuons"] = events.Muon[
            (events.Muon.pt >= 20.)
            & events.Muon.tightId
        ]

        events = events[
            (ak.num(events.goodmuons) == 2)
            & (ak.sum(events.goodmuons.charge, axis=1) == 0)
        ]
        cutflow["ossf"] = ak.num(events, axis=0)
        
        # add first and second muon p4 in every event together
        events["zcand"] = events.goodmuons[:, 0] + events.goodmuons[:, 1]

        # require trigger
        events = events[
            # https://twiki.cern.ch/twiki/bin/view/CMS/MuonHLT2018
            events.HLT.Mu17_TrkIsoVVL_Mu8_TrkIsoVVL_DZ_Mass3p8
        ]
        weight = 1 * ak.ones_like(events.event) if isRealData else events.genWeight
        cutflow["trigger"] = ak.num(events, axis=0)

        h = hist.Hist.new.Reg(120, 0., 120., label=r"$m_{\mu\mu}$ [GeV]").Weight()


        return {
            dataset: {
                "entries": ak.num(events, axis=0),
                "sumw": sumw,
                "cutflow": cutflow,
                "mass": h.fill(events.zcand.mass, weight=weight)
            }
        }

    def postprocess(self, accumulator):
        return accumulator

In [ ]:
from dask.distributed import Client, LocalCluster

# Configure your hardware footprint
cluster = LocalCluster(
    n_workers=4,            # Number of separate Python processes
    threads_per_worker=1,   # Threads per process (keep low to avoid GIL contention)
    memory_limit="1GB"      # Maximum RAM allocated to each worker process
)

# Connect your client session to the explicit cluster
client = Client(cluster)

In [ ]:
client

In [ ]:
run = processor.Runner(
    executor = processor.DaskExecutor(client=client, compression=None),
    schema=NanoAODSchema,
    chunksize=100_000,
    skipbadfiles=True,
    savemetrics=True,
    maxchunks=7,
)
small_result, small_report = run(
    initial_fileset,
    processor_instance=MyZPeak(),
)